# Proposal Automation RAG System

End-to-end pipeline:
- Ingest research papers
- Chunk + embed
- Store in FAISS
- Generate proposal ideas
- Generate full proposal
- Evaluate against NOFO

In [1]:
# @title This notebook assumes all dependencies are installed from requirements.txt prior to execution. Code is below (commented)
# Run in Bash: requirements.txt
# pip install -r requirements.txt

In [2]:
# Built-in dependencies
import os
import json
import zipfile

# Third-party dependencies
import tiktoken

# LangChain
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

In [3]:
PROJECT_ROOT = "."
CONFIG_PATH = f"{PROJECT_ROOT}/config.json"
ZIP_PATH = f"{PROJECT_ROOT}/Project_Files.zip"
EXTRACT_DIR = f"{PROJECT_ROOT}/Project_Files"

In [4]:
# Unzips Project_Files.zip

os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

print("All extracted files:")

for root, dirs, files in os.walk(EXTRACT_DIR):
    for file in files:
        print(os.path.join(root, file))


All extracted files:
./Project_Files/Project_Files/NOFO.pdf
./Project_Files/Project_Files/Research Proposal Template.pdf
./Project_Files/Project_Files/ResearchPapers.zip
./Project_Files/Project_Files/For Reference - Generated Sample Reseach Proposals/P4 - Personalized Mental Health Interventions Using Neural Data.pdf
./Project_Files/Project_Files/For Reference - Generated Sample Reseach Proposals/P1 - Enhancing Mental Health Engagement through Adaptive Social Media Interventions.pdf
./Project_Files/Project_Files/For Reference - Generated Sample Reseach Proposals/P2 - Neurofeedback-Driven Digital Interventions for Substance Use Disorders.pdf
./Project_Files/Project_Files/For Reference - Generated Sample Reseach Proposals/P5 - Social Network Analysis for Suicide Prevention Messaging.pdf
./Project_Files/Project_Files/For Reference - Generated Sample Reseach Proposals/P3 - AI-Enhanced Misinformation Countermeasures in Mental Health.pdf
./Project_Files/Papers/Misinformation_Simulation.pdf
.

In [53]:
# @title Load`config.json`(API Keys)
# Refactored this to avoid silent failures

api_key = os.getenv("OPENAI_API_KEY")
base_url = os.getenv("OPENAI_API_BASE")

# Fallback to config.json ONLY if env vars not set
if not api_key or not base_url:
    try:
        with open(CONFIG_PATH, "r") as f:
            config = json.load(f)

        api_key = api_key or config.get("API_KEY")
        base_url = base_url or config.get("OPENAI_API_BASE")

    except FileNotFoundError:
        pass

# Final validation
if not api_key:
    raise ValueError("OPENAI_API_KEY not found in environment or config.json")

if not base_url:
    base_url = "https://api.openai.com/v1"  # safe default

# Set environment (for downstream libs)
os.environ["OPENAI_API_KEY"] = api_key
os.environ["OPENAI_BASE_URL"] = base_url

print("Config loaded securely")



Config loaded securely


In [6]:
# @title Defining the LLM Model - Use `gpt-4o-mini` Model   #  gpt-4o for Production

try:
    from langchain_openai import ChatOpenAI
    print("langchain_openai already installed")
except ImportError:
    !pip install langchain-openai

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("LLM created using model: gpt-4o-mini")



langchain_openai already installed
LLM created using model: gpt-4o-mini


In [7]:
# Set PDF_PATH
PDF_PATH = None

for root, dirs, files in os.walk(EXTRACT_DIR):
    for file in files:
        if file == "NOFO.pdf":
            PDF_PATH = os.path.join(root, file)

if not PDF_PATH:
    raise FileNotFoundError("NOFO.pdf not found")

print("Using:", PDF_PATH)

Using: ./Project_Files/Project_Files/NOFO.pdf


In [8]:
# Reading the NOFO Document

pdf_loader = PyPDFLoader(PDF_PATH)
NOFO_pdf = pdf_loader.load()


In [9]:
# DEBUGGING: Verifying ingestion of PDF
print(len(NOFO_pdf))
print(NOFO_pdf[0].page_content[:500])

11
Department of Health and Human Services
Part 1. Overview Information
Key Dates
The following table includes NIH standard due dates (https://grants.nih.gov/grants/how-to-apply-application-guide/due-dates-and-submission-policies/due-dates.htm) marked with anasterisk.
Application Due Dates Review and Award Cycles
New
Renewal /Resubmission /Revision (asallowed)
AIDS -New/Renewal/Resubmission/Revision,as allowedScientiﬁc MeritReviewAdvisory CouncilReviewEarliest Start Date
February 05, 2025 *March 05


In [10]:
# DEBUGGING: Inspection of NOFO.pdf
for i, doc in enumerate(NOFO_pdf[:3]):
    print(f"\n--- Page {i+1} ---")
    print(doc.page_content[:300])


--- Page 1 ---
Department of Health and Human Services
Part 1. Overview Information
Key Dates
The following table includes NIH standard due dates (https://grants.nih.gov/grants/how-to-apply-application-guide/due-dates-and-submission-policies/due-dates.htm) marked with anasterisk.
Application Due Dates Review and A

--- Page 2 ---
June 05, 2026 * July 05, 2026 *Not Applicable November 2026January 2027April 2027
October 05, 2026 *November 05, 2026 *Not Applicable March 2027May 2027July 2027
February 05, 2027 *March 05, 2027 *Not Applicable July 2027October 2027December 2027
June 05, 2027 * July 05, 2027 *Not Applicable Novembe

--- Page 3 ---
mental health. As digital health technology for mental health is being increasingly used to provide standalone self-managed interventions and/or to supplement in-person treatment, well-designed pragmatic research is necessary to evaluate the eﬀectiveness of existing digital health technology, optimi


In [11]:
# nofo_text = "\n".join([doc.page_content for doc in NOFO_pdf])
nofo_text = "\n".join([doc.page_content for doc in NOFO_pdf[:2]])   # More efficient
print(len(nofo_text))
print('=' * 50)
print(nofo_text[:500])


10706
Department of Health and Human Services
Part 1. Overview Information
Key Dates
The following table includes NIH standard due dates (https://grants.nih.gov/grants/how-to-apply-application-guide/due-dates-and-submission-policies/due-dates.htm) marked with anasterisk.
Application Due Dates Review and Award Cycles
New
Renewal /Resubmission /Revision (asallowed)
AIDS -New/Renewal/Resubmission/Revision,as allowedScientiﬁc MeritReviewAdvisory CouncilReviewEarliest Start Date
February 05, 2025 *March 05


In [12]:
topic_extraction_prompt = """
Read the following Notice of Funding Opportunity (NOFO) and extract the primary funding topic.

Return ONLY the funding topic as a short phrase.
Do not include any explanation, punctuation, or additional words.

NOFO:
{nofo_text}

"""
final_prompt = topic_extraction_prompt.format(nofo_text=nofo_text)
print(topic_extraction_prompt)



Read the following Notice of Funding Opportunity (NOFO) and extract the primary funding topic.

Return ONLY the funding topic as a short phrase.
Do not include any explanation, punctuation, or additional words.

NOFO:
{nofo_text}




In [13]:
# Finding the topic for which the Funding is been given
topic_extraction = llm.invoke(final_prompt)
topic = topic_extraction.content.strip()
print(f"Topic: {topic}")


Topic: Digital mental health interventions


In [14]:
# DEBUGGING - Keep for future Debugging
# print("PROJECT_ROOT:", PROJECT_ROOT)
# print("Full ZIP path:", RESEARCH_ZIP_PATH)

# print("\nCurrent working dir:", os.getcwd())
# print("\nFiles in cwd:", os.listdir("."))


In [15]:
# Unzipping Project_Files
if os.path.exists(f"{PROJECT_ROOT}/Project_Files"):
    print("\nFiles in Project_Files:", os.listdir(f"{PROJECT_ROOT}/Project_Files"))
else:
    print("\nProject_Files folder NOT FOUND")


Files in Project_Files: ['Project_Files', 'Papers']


In [16]:
# Unzipping the "Database" which is currently PDF files contained in Research Papers
# Update zip file path and extarct it in the contents folder only

PROJECT_ROOT = os.getcwd()

RESEARCH_ZIP_PATH = f"{PROJECT_ROOT}/Project_Files/Project_Files/ResearchPapers.zip"
RESEARCH_EXTRACT_DIR = f"{PROJECT_ROOT}/Project_Files"

if not os.path.exists(RESEARCH_ZIP_PATH):
    raise FileNotFoundError(f"ZIP not found: {RESEARCH_ZIP_PATH}")

with zipfile.ZipFile(RESEARCH_ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(RESEARCH_EXTRACT_DIR)

print(os.listdir(RESEARCH_EXTRACT_DIR))

['Project_Files', 'Papers']


In [17]:
# Count of files unzipped

ROOT_PATH = PROJECT_ROOT

file_count = 0
all_files = []

for root, dirs, files in os.walk(ROOT_PATH):
    for file in files:
        full_path = os.path.join(root, file)
        all_files.append(full_path)
        file_count += 1

print(f"Total files found: {file_count}")



Total files found: 147


In [18]:
# Unzip ResearchPapers.zip

RESEARCH_ZIP_PATH = None

for root, dirs, files in os.walk(PROJECT_ROOT):
    for file in files:
        if file == "ResearchPapers.zip":
            RESEARCH_ZIP_PATH = os.path.join(root, file)

if not RESEARCH_ZIP_PATH:
    print("ResearchPapers.zip not found — likely already extracted or stored elsewhere.")
else:
    RESEARCH_EXTRACT_DIR = f"{PROJECT_ROOT}/Project_Files"
    with zipfile.ZipFile(RESEARCH_ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(RESEARCH_EXTRACT_DIR)
    print("ResearchPapers.zip extracted")

ResearchPapers.zip extracted


In [19]:
# Verifies if the file "ALL18.pdf is among the files unzipped
import glob

SEARCH_PATTERN = f"{PROJECT_ROOT}/**/ALL18.pdf"
matches = glob.glob(SEARCH_PATTERN, recursive=True)

if matches:
    print("FOUND:")
    for match in matches:
        print(match)
else:
    print("ALL18.pdf not found.")


FOUND:
/content/Project_Files/Papers/ALL18.pdf


In [20]:
# Examine the first 500 charcters of file: ALL18.pdf

all18_path = matches[0]
loader = PyPDFLoader(all18_path)
ALL18_docs = loader.load()

print(len(ALL18_docs))
print(ALL18_docs[0].page_content[:500])


13
APPLIED LANGUAGE LEARNING                      2008                  VOLUME  18        NUMBERS  1 & 2
  Applied
   Language
    Learning
VOLUME 18 ·  NUMBERS 1 & 2


In [21]:
# DEBUGGING: Inspection of NOFO.pdf
for i, doc in enumerate(NOFO_pdf[:3]):
    print(f"\n--- Page {i+1} ---")
    print(doc.page_content[:300])


--- Page 1 ---
Department of Health and Human Services
Part 1. Overview Information
Key Dates
The following table includes NIH standard due dates (https://grants.nih.gov/grants/how-to-apply-application-guide/due-dates-and-submission-policies/due-dates.htm) marked with anasterisk.
Application Due Dates Review and A

--- Page 2 ---
June 05, 2026 * July 05, 2026 *Not Applicable November 2026January 2027April 2027
October 05, 2026 *November 05, 2026 *Not Applicable March 2027May 2027July 2027
February 05, 2027 *March 05, 2027 *Not Applicable July 2027October 2027December 2027
June 05, 2027 * July 05, 2027 *Not Applicable Novembe

--- Page 3 ---
mental health. As digital health technology for mental health is being increasingly used to provide standalone self-managed interventions and/or to supplement in-person treatment, well-designed pragmatic research is necessary to evaluate the eﬀectiveness of existing digital health technology, optimi


In [22]:
# @title EXPERIMENT WITH chunk_size (default=800) and chunk_overlap (default=150)
# Several Files - which takes this a few minutes to run
from langchain_community.document_loaders import PyPDFDirectoryLoader

PAPERS_DIR = f"{EXTRACT_DIR}/Papers"

loader = PyPDFDirectoryLoader(PAPERS_DIR)
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n", "\n", ".", " ", ""]
)

chunks = text_splitter.split_documents(documents)

print('=' * 50)
print(f"Total chunks: {len(chunks)}")


Total chunks: 9109


# 2.2 Create embeddings

In [23]:
# Create embeddings
# UPDATE HuggingFaceEmbeddings

from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("HuggingFaceEmbeddings Imported")

/tmp/ipykernel_14858/18127567.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


HuggingFaceEmbeddings Imported


In [24]:
# @title Store Chunks in FAISS INDEX - DELETE this comment
# Store chunks in FAISS index

FAISS_PATH = "/content/faiss_index"

if os.path.exists(FAISS_PATH):
    print("Loading existing FAISS index...")
    vectorstore = FAISS.load_local(
        FAISS_PATH,
        embeddings,
        allow_dangerous_deserialization=True
    )
else:
    print("Creating new FAISS index...")
    vectorstore = FAISS.from_documents(
        documents=chunks,
        embedding=embeddings
    )
    # Save index to disk
    vectorstore.save_local(FAISS_PATH)

print("FAISS index saved")


Loading existing FAISS index...
FAISS index saved


In [25]:
# Load FAISS index

vectorstore = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)
print("FAISS Index Loaded")

FAISS Index Loaded


In [26]:
# Retrieve top-K relevant chunks by topic


top_k = 50   # EXPERIIMENT with # of chunks and note effect on relevant content
docs = vectorstore.similarity_search("Digital Mental Health Interventions", k=top_k)

print(f"Total results returned: {len(docs)}")
print('=' * 40)

for i, doc in enumerate(docs[:3], 1):
# for i, doc in enumerate(docs, 1):       # PRINT ALL
    print(f"\n--- Result {i} ---")

    print("Source:", doc.metadata.get("source"))
    print(doc.page_content[:400])


Total results returned: 50

--- Result 1 ---
Source: Project_Files/Papers/Social Media Mental Health Final.pdf
ASONAM ’23, November 6-9, 2023, Kusadasi, Turkey © 2023 Association for Computing Machinery. 
ACM ISBN 979-8-4007-0409-3/23/11. . . $15.00 https://doi.org/10.1145/3625007.3627490 
Fragile Minds: Exploring the Link Between Social 
Media and Young Adult Mental Health 
 
 
 
 
Ben Cohen 
Whiting School of Engineering 
Johns Hopkins University 
Baltimore, MD, USA 
bcohen49@jhu.edu 
 
Ian McCulloh 
Joh

--- Result 2 ---
Source: Project_Files/Papers/Social Media Mental Health Final.pdf
in young adult behavior and interaction with social media 
between 2016 and the wide-spread adoption of short -video 
content is alleviating some of the mental health concerns. It is 
also possible that there is an inherent response bias, with 
individuals who do feel that their mental health has been harmed 
by using social media less likely to respond to our survey 
invitation. Our finding sugge

--

In [27]:
# Determine relevant documents
from collections import defaultdict
document_dict=defaultdict(int)

for i, doc in enumerate(docs, 1):
    #  +=1 is for determining which PDFs contributed the most matching chunks (a crude relevance signal).
    document_dict[doc.metadata.get("source")] +=1

relevant_files_sorted = sorted(document_dict.items(), key=lambda x: x[1], reverse=True)


print(f"Unique relevant documents: {len(relevant_files_sorted)}")
for path, count in relevant_files_sorted:
    print(f"{count:>2}  {path}")



Unique relevant documents: 16
13  Project_Files/Papers/Social Media Mental Health Final.pdf
 9  Project_Files/Papers/NAP Behavioral Sci Intel.pdf
 5  Project_Files/Papers/HIV.pdf
 4  Project_Files/Papers/Arrow White Paper DExTra.pdf
 3  Project_Files/Papers/YouTube-COVID.pdf
 3  Project_Files/Papers/23-US-DHS-001.pdf
 2  Project_Files/Papers/Misinformation_Simulation.pdf
 2  Project_Files/Papers/improving-decision-support-for-organ-transplant.pdf
 2  Project_Files/Papers/NeuroCogInfluence.pdf
 1  Project_Files/Papers/cycon-final-draft.pdf
 1  Project_Files/Papers/White Paper Brain Gaze.pdf
 1  Project_Files/Papers/COVID Bayesian Data Aug.pdf
 1  Project_Files/Papers/Social_Det_COVID_Mortality.pdf
 1  Project_Files/Papers/Savas.pdf
 1  Project_Files/Papers/CausalOrgInorgContent.pdf
 1  Project_Files/Papers/WEIRD.pdf


In [28]:
# Generate summaries for all the relevant documents

summary_prompt = f"""


You are summarizing a research paper to support drafting a grant proposal aligned to this NOFO topic:
Digital mental health interventions

Rules:
- Use ONLY the provided paper text. Do not guess.
- If something is not present in the text, write "Not stated".
- Be concise and factual.

Return the summary in EXACTLY this format:

TITLE: <title or Not stated>
ONE-SENTENCE SUMMARY: <1 sentence>
KEY POINTS (3-6 bullets):
- ...
METHOD / DATA: <1-2 sentences or Not stated>
WHY THIS IS RELEVANT TO DIGITAL MENTAL HEALTH (1-2 sentences): <...>

"""
print(summary_prompt)

# Keep and append to prompt if I go a different direction with this prompt
# PAPER TEXT:
# {paper_text}




You are summarizing a research paper to support drafting a grant proposal aligned to this NOFO topic:
Digital mental health interventions

Rules:
- Use ONLY the provided paper text. Do not guess.
- If something is not present in the text, write "Not stated".
- Be concise and factual.

Return the summary in EXACTLY this format:

TITLE: <title or Not stated>
ONE-SENTENCE SUMMARY: <1 sentence>
KEY POINTS (3-6 bullets):
- ...
METHOD / DATA: <1-2 sentences or Not stated>
WHY THIS IS RELEVANT TO DIGITAL MENTAL HEALTH (1-2 sentences): <...>




In [54]:
# Load `config.json`
# Refactored to avoid silent failures

# Try environment first (safe for GitHub)
api_key = os.getenv("OPENAI_API_KEY")
base_url = os.getenv("OPENAI_API_BASE")

# Fallback to config.json if needed
if not api_key:
    try:
        with open(CONFIG_PATH, "r") as f:
            config = json.load(f)

        api_key = config.get("API_KEY")
        base_url = base_url or config.get("OPENAI_API_BASE")

    except FileNotFoundError:
        pass

# Validate
if not api_key:
    raise ValueError("OPENAI_API_KEY not found (env or config.json)")

# Default base URL if missing
if not base_url:
    base_url = "https://api.openai.com/v1"

# Set env for downstream use
os.environ["OPENAI_API_KEY"] = api_key
os.environ["OPENAI_BASE_URL"] = base_url

print("Config loaded safely")


Config loaded safely


In [30]:
# @title Defining the LLM Model - Use `gpt-4o-mini` Model     #  gpt-4o for Production
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("LLM created using model: gpt-4o-mini")

LLM created using model: gpt-4o-mini


In [31]:
# Encoding

documents = []


# Defining the max tokens to avoid error for context being too long

encoding = tiktoken.encoding_for_model("gpt-4o-mini")
# encoding = tiktoken.encoding_for_model("gpt-4o")
MAX_TOKENS = 127500

for file_path in list(document_dict.keys()):

  # Extract filename
  filename = file_path.split("/")[-1]

  # Load PDF
  docs = PyPDFLoader(file_path,mode="single").load()

  # extract pages
  # pages = docs[0].page_content
  pages = "\n".join([d.page_content for d in docs])

  # combining the prompt with the pages of the research paper within the context length
  available_tokens = MAX_TOKENS - len(encoding.encode(summary_prompt))
  truncated_pages = encoding.decode(encoding.encode(pages)[:available_tokens])
  full_prompt = summary_prompt + truncated_pages

  # Calling the LLM
  response = llm.invoke(full_prompt)

  #  If paper is relevant adding it to the documents variable
  documents.append({ 'title': filename, 'llm_response': response.content, 'file_path':file_path})



In [32]:
# check len() for DEBUGGING

print("Total summarized documents:", len(documents))

Total summarized documents: 16


In [33]:
# Generating 5 Ideas


gen_idea_prompt = """
You are an expert research proposal strategist helping prepare a grant proposal response to an NIH Notice of Funding Opportunity (NOFO).

Your task is to generate exactly 5 research proposal ideas based ONLY on the filtered research paper summaries provided below.

Each idea must follow this exact structure:

Idea X: [Concise Title of the Project Idea]
Description: [Brief and targeted description summarizing the objectives, innovative elements, scientific rationale, and anticipated impact.]
Citation: [Author(s), Year or Paper Title]
NOFO Alignment: [List two or more specific NOFO requirements that this idea directly addresses]
File Path of the Research Paper: [Exact file path, ending in .pdf]
---

Instructions:
1. Generate exactly 5 distinct proposal ideas.
2. Use only the provided filtered research summaries and file paths.
3. Each idea must be grounded in one of the filtered papers.
4. The Citation must reference the source paper using either the paper title or authors/year if clearly available.
5. The File Path must be copied exactly as provided and must end in .pdf.
6. The NOFO Alignment must include at least two specific requirements, priorities, or goals directly addressed by the idea.
7. Keep each idea concise, specific, scientifically credible, and aligned to the NOFO topic.
8. Do not invent file paths, citations, or unsupported claims.
9. Use the delimiter --- exactly after each idea.

Sample Output Format:

Idea 1: Adaptive Mobile Support for Suicide Risk Monitoring
Description: This project will test a mobile-based digital mental health intervention that uses real-time behavioral and self-reported data to identify elevated suicide risk and deliver timely adaptive support. The study emphasizes scalable deployment, rapid testing, and improved engagement among vulnerable populations.
Citation: Example Paper Title or Author et al., 2023
NOFO Alignment: Digital mental health interventions; real-time data collection and feedback; improving reach and effectiveness; support for vulnerable populations
File Path of the Research Paper: {EXTRACT_DIR}/Papers/example.pdf
---

Now generate 5 proposal ideas from the filtered research summaries below:

{document_summaries}
"""
print(gen_idea_prompt)


You are an expert research proposal strategist helping prepare a grant proposal response to an NIH Notice of Funding Opportunity (NOFO).

Your task is to generate exactly 5 research proposal ideas based ONLY on the filtered research paper summaries provided below.

Each idea must follow this exact structure:

Idea X: [Concise Title of the Project Idea]
Description: [Brief and targeted description summarizing the objectives, innovative elements, scientific rationale, and anticipated impact.]
Citation: [Author(s), Year or Paper Title]
NOFO Alignment: [List two or more specific NOFO requirements that this idea directly addresses]
File Path of the Research Paper: [Exact file path, ending in .pdf]
---

Instructions:
1. Generate exactly 5 distinct proposal ideas.
2. Use only the provided filtered research summaries and file paths.
3. Each idea must be grounded in one of the filtered papers.
4. The Citation must reference the source paper using either the paper title or authors/year if clear

In [34]:
# Generate document_summaries - Addedd for Debugging/Improvement
document_summaries = ""

for doc in documents:
    document_summaries += f"""
Title: {doc['title']}
Summary: {doc['llm_response']}
File Path: {doc['file_path']}
"""
print(document_summaries)


Title: Social Media Mental Health Final.pdf
Summary: TITLE: Fragile Minds: Exploring the Link Between Social Media and Young Adult Mental Health  
ONE-SENTENCE SUMMARY: This paper investigates the correlation between social media usage and mental health concerns among college students, finding moderate correlations but suggesting that social media use alone cannot fully explain mental health outcomes.  
KEY POINTS (3-6 bullets):  
- The study involved a random sample of 254 undergraduate students and examined correlations between social media use, loneliness, and fear of missing out (FoMO).  
- Findings indicate a weak correlation between social media use and loneliness, with personality traits moderating this relationship.  
- The rise of video-based platforms like TikTok and changes in Instagram's focus may be influencing mental health outcomes differently than previous studies suggested.  
- The research highlights the need for further investigation into the impact of social media 

In [35]:
#
full_prompt = gen_idea_prompt.format(
    document_summaries=document_summaries,
    EXTRACT_DIR=EXTRACT_DIR
)
print(full_prompt)



You are an expert research proposal strategist helping prepare a grant proposal response to an NIH Notice of Funding Opportunity (NOFO).

Your task is to generate exactly 5 research proposal ideas based ONLY on the filtered research paper summaries provided below.

Each idea must follow this exact structure:

Idea X: [Concise Title of the Project Idea]
Description: [Brief and targeted description summarizing the objectives, innovative elements, scientific rationale, and anticipated impact.]
Citation: [Author(s), Year or Paper Title]
NOFO Alignment: [List two or more specific NOFO requirements that this idea directly addresses]
File Path of the Research Paper: [Exact file path, ending in .pdf]
---

Instructions:
1. Generate exactly 5 distinct proposal ideas.
2. Use only the provided filtered research summaries and file paths.
3. Each idea must be grounded in one of the filtered papers.
4. The Citation must reference the source paper using either the paper title or authors/year if clear

In [36]:
ideas = llm.invoke(full_prompt)
print(ideas.content)

Idea 1: Understanding Social Media's Impact on Mental Health
Description: This project aims to explore the nuanced relationship between social media usage and mental health outcomes among college students, focusing on the moderating effects of personality traits. By conducting a larger-scale survey and incorporating qualitative interviews, the study will provide insights into how different social media platforms affect loneliness and mental well-being, ultimately informing the design of targeted digital mental health interventions.
Citation: Fragile Minds: Exploring the Link Between Social Media and Young Adult Mental Health
NOFO Alignment: Investigating mental health outcomes; targeting vulnerable populations; developing digital mental health interventions
File Path of the Research Paper: Project_Files/Papers/Social Media Mental Health Final.pdf
---

Idea 2: Enhancing Public Health Communication through YouTube Engagement Analysis
Description: This research will analyze the engagement

In [37]:
# Applying Markdown, code refactored to aid in formatting

import re
from IPython.display import Markdown, display

formatted_ideas = ideas.content

# Make each idea title a smaller markdown heading
formatted_ideas = re.sub(r"^Idea (\d+:.*)$", r"### Idea \1", formatted_ideas, flags=re.MULTILINE)

# Bold ONLY the field labels
formatted_ideas = re.sub(r"^Description:", r"**Description:**", formatted_ideas, flags=re.MULTILINE)
formatted_ideas = re.sub(r"^Citation:", r"**Citation:**", formatted_ideas, flags=re.MULTILINE)
formatted_ideas = re.sub(r"^NOFO Alignment:", r"**NOFO Alignment:**", formatted_ideas, flags=re.MULTILINE)
formatted_ideas = re.sub(r"^File Path of the Research Paper:", r"**File Path of the Research Paper:**", formatted_ideas, flags=re.MULTILINE)

# Ensure horizontal rule has spacing around it
formatted_ideas = formatted_ideas.replace("---", "\n\n---\n\n")  # May have to lookup on ChatGPT if scope changes

display(Markdown(formatted_ideas))


### Idea 1: Understanding Social Media's Impact on Mental Health
**Description:** This project aims to explore the nuanced relationship between social media usage and mental health outcomes among college students, focusing on the moderating effects of personality traits. By conducting a larger-scale survey and incorporating qualitative interviews, the study will provide insights into how different social media platforms affect loneliness and mental well-being, ultimately informing the design of targeted digital mental health interventions.
**Citation:** Fragile Minds: Exploring the Link Between Social Media and Young Adult Mental Health
**NOFO Alignment:** Investigating mental health outcomes; targeting vulnerable populations; developing digital mental health interventions
**File Path of the Research Paper:** Project_Files/Papers/Social Media Mental Health Final.pdf


---



### Idea 2: Enhancing Public Health Communication through YouTube Engagement Analysis
**Description:** This research will analyze the engagement patterns and polarization of COVID-19-related YouTube videos to develop effective public health communication strategies. By understanding how different types of videos are received by audiences, the project aims to create guidelines for producing less polarizing and more engaging health content, which can be crucial during health crises.
**Citation:** Approaches for Quantifying Video Prominence, Narratives, & Discussion: Engagement on COVID-19 Related YouTube Videos
**NOFO Alignment:** Improving public health communication; addressing misinformation; enhancing engagement in health interventions
**File Path of the Research Paper:** Project_Files/Papers/YouTube-COVID.pdf


---



### Idea 3: Non-Invasive Dopamine Measurement for Mental Health Diagnostics
**Description:** This project proposes the development of a non-invasive AI tool to measure synaptic dopamine levels using webcam technology, aimed at enhancing diagnostic capabilities for mental health conditions related to dopamine deficiencies. By integrating eye-tracking and facial expression recognition, the tool will provide real-time insights into mental health status, potentially transforming treatment approaches for conditions like depression and addiction.
**Citation:** White Paper for Non-Invasive Synaptic Dopamine Measurement
**NOFO Alignment:** Innovative diagnostic tools for mental health; addressing addiction and neurological disorders; enhancing treatment strategies
**File Path of the Research Paper:** Project_Files/Papers/White Paper Brain Gaze.pdf


---



### Idea 4: Mitigating Misinformation's Impact on Mental Health
**Description:** This study will develop an agent-based simulation to analyze how different populations respond to misinformation, focusing on the psychological factors that influence these responses. By identifying cognitive biases and mental schemas, the project aims to create tailored digital mental health interventions that can effectively address misinformation-related stress and anxiety in diverse communities.
**Citation:** SIMULATING MISINFORMATION VULNERABILITIES WITH AGENT PERSONAS
**NOFO Alignment:** Addressing misinformation; developing targeted mental health interventions; understanding cognitive biases
**File Path of the Research Paper:** Project_Files/Papers/Misinformation_Simulation.pdf


---



### Idea 5: Analyzing the Effects of Online Interactions on Emotional Well-Being
**Description:** This research will investigate the causal effects of interactions between organic social media users and inorganic accounts on emotional outcomes. By employing a Neyman-Rubin causal model, the study aims to quantify how these interactions influence sentiments like anger and fear, ultimately informing the development of digital mental health interventions that can mitigate negative emotional impacts from harmful online behaviors.
**Citation:** Causal Analysis of Online Outcomes of Interactions between Organic and Inorganic Accounts
**NOFO Alignment:** Understanding emotional impacts of online interactions; developing interventions for mental well-being; addressing harmful online behaviors
**File Path of the Research Paper:** Project_Files/Papers/CausalOrgInorgContent.pdf


---



#### Choosing 1 Idea and fetching details

In [38]:
# Choose 1 Idea and fetching details

# Modify the idea_number for choosing the different idea
idea_number = 5   # change the number if you wish to choose and generate the research proposal for another idea
chosen_idea = ideas.content.split("---")[idea_number -1].strip()    # added -1 - zero based ary

print('=' * 50)
print(chosen_idea)
print('=' * 50)



Idea 5: Analyzing the Effects of Online Interactions on Emotional Well-Being
Description: This research will investigate the causal effects of interactions between organic social media users and inorganic accounts on emotional outcomes. By employing a Neyman-Rubin causal model, the study aims to quantify how these interactions influence sentiments like anger and fear, ultimately informing the development of digital mental health interventions that can mitigate negative emotional impacts from harmful online behaviors.
Citation: Causal Analysis of Online Outcomes of Interactions between Organic and Inorganic Accounts
NOFO Alignment: Understanding emotional impacts of online interactions; developing interventions for mental well-being; addressing harmful online behaviors
File Path of the Research Paper: Project_Files/Papers/CausalOrgInorgContent.pdf


In [40]:
import re

# Use a regular expression to find the file path of the research paper
pattern = r"File Path of the Research Paper:\s*(.*\.pdf)"
#pattern = r"\*\*File Path of the Research Paper:\*\*\s*(.+)"

match = re.search(pattern, chosen_idea)

if match:
  idea_generated_from_research_paper = match.group(1).strip()
  print("Filepath : ", idea_generated_from_research_paper)
else:
  print("File Path of the Research Paper not found in the chosen idea.")

Filepath :  Project_Files/Papers/CausalOrgInorgContent.pdf


In [41]:
# Here we need to add the full papers instead of the summary

TEMPLATE_PATH = os.path.join(
    EXTRACT_DIR,
    "Project_Files",
    "Research Proposal Template.pdf"
)

if not os.path.exists(TEMPLATE_PATH):
    raise FileNotFoundError(f"Template not found: {TEMPLATE_PATH}")

print('=' * 50)
print(TEMPLATE_PATH)


./Project_Files/Project_Files/Research Proposal Template.pdf


In [42]:
research_proposal_template_prompt = f"""

You are preparing a structured research proposal.

Use the selected research idea and supporting research paper to generate a proposal that follows the EXACT structure and headings shown below.

Rules:
- Follow the section order EXACTLY as shown.
- Do NOT change section titles.
- Maintain numbering of sections.
- Write clear academic prose appropriate for a research proposal.
- Ground the content in the selected idea and cited research paper.
- Do not invent references beyond the provided research paper.
- Keep formatting consistent with the template.

Selected Idea:
{chosen_idea}

Source Research Paper Path:
{idea_generated_from_research_paper}

Now generate the proposal following this EXACT structure:

Community-Driven Digital Health Platforms for Mental Health Support

1. Introduction
Write a clear introduction explaining the importance of the problem, background context, and why digital mental health interventions are needed.

2. Problem Statement
Explain the core problem being addressed and list the research questions. Include 3 research questions relevant to the selected idea.

3. Objectives
Describe the main objective of the research followed by 3 specific objectives.

4. Preliminary Literature Review
Summarize relevant insights from the supporting research paper and explain how prior work motivates the proposed research.

5. Methodology
Explain the research design using a mixed-methods approach. Structure this section exactly as:

Phase 1: Platform Development
Phase 2: User Engagement and Data Collection
Phase 3: Evaluation and Analysis

Provide short bullet points for each phase.

References
Include the research paper used for generating this proposal.

6. Expected Outcomes
Explain the expected impact and contributions of the research.

7. Budget Justification
Provide a brief explanation of the types of costs required for the project.

8. Conclusion
Provide a short concluding paragraph summarizing the proposal and its potential impact.

Return only the proposal text.
Do not include explanations outside the proposal.
"""

print(research_proposal_template_prompt)



You are preparing a structured research proposal.

Use the selected research idea and supporting research paper to generate a proposal that follows the EXACT structure and headings shown below.

Rules:
- Follow the section order EXACTLY as shown.
- Do NOT change section titles.
- Maintain numbering of sections.
- Write clear academic prose appropriate for a research proposal.
- Ground the content in the selected idea and cited research paper.
- Do not invent references beyond the provided research paper.
- Keep formatting consistent with the template.

Selected Idea:
Idea 5: Analyzing the Effects of Online Interactions on Emotional Well-Being
Description: This research will investigate the causal effects of interactions between organic social media users and inorganic accounts on emotional outcomes. By employing a Neyman-Rubin causal model, the study aims to quantify how these interactions influence sentiments like anger and fear, ultimately informing the development of digital menta

In [43]:

proposal_prompt = research_proposal_template_prompt.format(
    chosen_idea=chosen_idea,
    idea_generated_from_research_paper=idea_generated_from_research_paper
)

proposal = llm.invoke(proposal_prompt)

print(proposal.content)

# Refactored from:
# research_plan = llm.invoke(research_proposal_template_prompt)

Community-Driven Digital Health Platforms for Mental Health Support

1. Introduction  
The rise of social media has transformed the way individuals interact, creating both opportunities and challenges for emotional well-being. As users engage with a mix of organic accounts—real users—and inorganic accounts—bots or automated profiles—the emotional consequences of these interactions have become increasingly significant. Understanding how these dynamics affect sentiments such as anger and fear is crucial for developing effective digital mental health interventions. Given the prevalence of harmful online behaviors, there is an urgent need to explore these interactions to inform strategies that can mitigate negative emotional impacts and promote mental well-being.

2. Problem Statement  
The core problem being addressed is the lack of understanding regarding the emotional effects of interactions between organic social media users and inorganic accounts. This research aims to answer the foll

In [44]:
import re
from IPython.display import Markdown, display

proposal_text = proposal.content

# Title (slightly larger + bold)
proposal_text = re.sub(
    r"^Community-Driven Digital Health Platforms for Mental Health Support",
    r"### **Community-Driven Digital Health Platforms for Mental Health Support**",
    proposal_text
)

# Bold section headings
proposal_text = re.sub(r"\n1\. Introduction", r"\n\n#### **1. Introduction**", proposal_text)
proposal_text = re.sub(r"\n2\. Problem Statement", r"\n\n#### **2. Problem Statement**", proposal_text)
proposal_text = re.sub(r"\n3\. Objectives", r"\n\n#### **3. Objectives**", proposal_text)
proposal_text = re.sub(r"\n4\. Preliminary Literature Review", r"\n\n#### **4. Preliminary Literature Review**", proposal_text)
proposal_text = re.sub(r"\n5\. Methodology", r"\n\n#### **5. Methodology**", proposal_text)
proposal_text = re.sub(r"\n6\. Expected Outcomes", r"\n\n#### **6. Expected Outcomes**", proposal_text)
proposal_text = re.sub(r"\n7\. Budget Justification", r"\n\n#### **7. Budget Justification**", proposal_text)
proposal_text = re.sub(r"\n8\. Conclusion", r"\n\n#### **8. Conclusion**", proposal_text)

# Convert bullet lists to numbered lists with indent
# proposal_text = re.sub(r"\n-\s*", r"\n    1. ", proposal_text)

display(Markdown(proposal_text))


### **Community-Driven Digital Health Platforms for Mental Health Support**


#### **1. Introduction**  
The rise of social media has transformed the way individuals interact, creating both opportunities and challenges for emotional well-being. As users engage with a mix of organic accounts—real users—and inorganic accounts—bots or automated profiles—the emotional consequences of these interactions have become increasingly significant. Understanding how these dynamics affect sentiments such as anger and fear is crucial for developing effective digital mental health interventions. Given the prevalence of harmful online behaviors, there is an urgent need to explore these interactions to inform strategies that can mitigate negative emotional impacts and promote mental well-being.


#### **2. Problem Statement**  
The core problem being addressed is the lack of understanding regarding the emotional effects of interactions between organic social media users and inorganic accounts. This research aims to answer the following questions:  
- How do interactions with inorganic accounts influence the emotional states of organic users?  
- What specific sentiments, such as anger and fear, are most affected by these interactions?  
- How can insights from these interactions inform the development of digital mental health interventions?


#### **3. Objectives**  
The main objective of this research is to investigate the causal effects of online interactions on emotional well-being, particularly focusing on the dynamics between organic and inorganic accounts.  
- Specific Objective 1: To quantify the emotional outcomes of interactions with inorganic accounts using a Neyman-Rubin causal model.  
- Specific Objective 2: To identify the specific sentiments that are most impacted by these interactions.  
- Specific Objective 3: To develop recommendations for digital mental health interventions based on the findings.


#### **4. Preliminary Literature Review**  
The supporting research paper, "Causal Analysis of Online Outcomes of Interactions between Organic and Inorganic Accounts," provides critical insights into the emotional consequences of online interactions. It highlights the complex relationship between user engagement and emotional outcomes, demonstrating that interactions with inorganic accounts can lead to heightened feelings of anger and fear among organic users. This prior work underscores the necessity of further investigation into these dynamics, motivating the proposed research to explore causal relationships and inform intervention strategies.


#### **5. Methodology**  
The research design will employ a mixed-methods approach structured as follows:  

Phase 1: Platform Development  
- Create a digital platform for users to report interactions with inorganic accounts.  
- Integrate tools for tracking emotional responses and sentiment analysis.  

Phase 2: User Engagement and Data Collection  
- Recruit organic social media users to participate in the study.  
- Collect qualitative and quantitative data on user interactions and emotional outcomes.  

Phase 3: Evaluation and Analysis  
- Analyze the data using the Neyman-Rubin causal model to assess emotional impacts.  
- Evaluate the findings to develop actionable recommendations for digital mental health interventions.  

References  
Causal Analysis of Online Outcomes of Interactions between Organic and Inorganic Accounts. Project_Files/Papers/CausalOrgInorgContent.pdf


#### **6. Expected Outcomes**  
The expected impact of this research includes a deeper understanding of how online interactions affect emotional well-being, leading to the development of targeted digital mental health interventions. By quantifying the emotional consequences of these interactions, the research aims to provide valuable insights that can help mitigate negative emotional impacts and promote healthier online environments.


#### **7. Budget Justification**  
The project will require funding for platform development, user recruitment, data collection tools, and analysis software. Additionally, costs will include personnel for data analysis and participant support, as well as dissemination of findings through publications and community outreach.


#### **8. Conclusion**  
In conclusion, this proposal outlines a critical investigation into the effects of online interactions on emotional well-being, with the potential to inform effective digital mental health interventions. By addressing the emotional impacts of interactions between organic and inorganic accounts, this research aims to contribute significantly to the field of digital mental health and promote healthier online experiences for users.

In [47]:
# PICKUP HERE
# **Create PDF of Research Proposal**

import markdown
from weasyprint import HTML

html = markdown.markdown(proposal.content)
HTML(string=html).write_pdf("Reseach Proposal First Draft.pdf")


DEBUG:fontTools.ttLib.ttFont:Reading 'maxp' table from disk
DEBUG:fontTools.ttLib.ttFont:Decompiling 'maxp' table
DEBUG:fontTools.subset.timer:Took 0.002s to load 'maxp'
DEBUG:fontTools.subset.timer:Took 0.000s to prune 'maxp'
INFO:fontTools.subset:maxp pruned
DEBUG:fontTools.ttLib.ttFont:Reading 'cmap' table from disk
DEBUG:fontTools.ttLib.ttFont:Decompiling 'cmap' table
DEBUG:fontTools.ttLib.ttFont:Reading 'post' table from disk
DEBUG:fontTools.ttLib.ttFont:Decompiling 'post' table
DEBUG:fontTools.subset.timer:Took 0.007s to load 'cmap'
DEBUG:fontTools.subset.timer:Took 0.000s to prune 'cmap'
INFO:fontTools.subset:cmap pruned
INFO:fontTools.subset:fpgm dropped
INFO:fontTools.subset:prep dropped
INFO:fontTools.subset:cvt  dropped
DEBUG:fontTools.subset.timer:Took 0.000s to load 'post'
DEBUG:fontTools.subset.timer:Took 0.000s to prune 'post'
INFO:fontTools.subset:post pruned
DEBUG:fontTools.ttLib.ttFont:Reading 'glyf' table from disk
DEBUG:fontTools.ttLib.ttFont:Decompiling 'glyf' tabl

In [48]:
# Proposal Evaluation Against NOFO Criteria

proposal_evaluation_prompt = """
You are acting as an expert research grant reviewer evaluating a proposal submitted to a Notice of Funding Opportunity (NOFO).

Your task is to evaluate the research proposal using the following criteria:

1. Innovation
2. Significance
3. Approach
4. Investigator Expertise

For EACH criterion:
- Assign a score from 1 (Poor) to 5 (Excellent).
- Provide a brief justification.
- Identify one key strength.
- Identify one key weakness.
- Provide one recommendation for improvement.

Use the NOFO text to judge how well the proposal aligns with the goals and priorities.

NOFO TEXT:
{nofo_text}

RESEARCH PROPOSAL:
{generated_proposal}

Return the evaluation STRICTLY in the following JSON format:

[
  {{
    "name": "Innovation",
    "justification": "<Justification>",
    "score": <1-5>,
    "strengths": "<Strength>",
    "weaknesses": "<Weakness>",
    "recommendations": "<Recommendation>"
  }},
  {{
    "name": "Significance",
    "justification": "<Justification>",
    "score": <1-5>,
    "strengths": "<Strength>",
    "weaknesses": "<Weakness>",
    "recommendations": "<Recommendation>"
  }},
  {{
    "name": "Approach",
    "justification": "<Justification>",
    "score": <1-5>,
    "strengths": "<Strength>",
    "weaknesses": "<Weakness>",
    "recommendations": "<Recommendation>"
  }},
  {{
    "name": "Investigator Expertise",
    "justification": "<Justification>",
    "score": <1-5>,
    "strengths": "<Strength>",
    "weaknesses": "<Weakness>",
    "recommendations": "<Recommendation>"
  }}
]

Return ONLY valid JSON. Do not include explanations outside the JSON.
"""

In [49]:
# .join pages

PDF_PATH = os.path.join(
    EXTRACT_DIR,
    "Project_Files",
    "NOFO.pdf"
)

if not os.path.exists(PDF_PATH):
    raise FileNotFoundError(f"NOFO not found: {PDF_PATH}")

pdf_loader = PyPDFLoader(PDF_PATH)
NOFO_pdf = pdf_loader.load()

nofo_text = "\n".join([doc.page_content for doc in NOFO_pdf[:2]])

In [50]:
# Evaluate via LLM

evaluation_prompt = proposal_evaluation_prompt.format(
    nofo_text=nofo_text,
    generated_proposal=proposal.content
)

evaluation = llm.invoke(evaluation_prompt)

print(evaluation.content)


# Changed this. Keep for reference and DEBUGGING
#eval_response = llm.invoke(evaluation_prompt)

[
  {
    "name": "Innovation",
    "justification": "The proposal introduces a novel approach by focusing on the emotional effects of interactions between organic and inorganic social media accounts, which is a relatively unexplored area in digital mental health research.",
    "score": 4,
    "strengths": "The innovative focus on the dynamics of social media interactions provides a fresh perspective on mental health interventions.",
    "weaknesses": "While the concept is innovative, the application of existing methodologies may limit the potential for groundbreaking findings.",
    "recommendations": "Consider incorporating more cutting-edge technologies or methodologies, such as machine learning for sentiment analysis, to enhance the innovative aspect."
  },
  {
    "name": "Significance",
    "justification": "The research addresses a critical gap in understanding how social media interactions impact emotional well-being, which is highly relevant given the increasing use of digita

In [51]:
import re

raw_json = evaluation.content.strip()

# Remove markdown code fences if present
raw_json = re.sub(r"^```json\s*", "", raw_json)
raw_json = re.sub(r"^```\s*", "", raw_json)
raw_json = re.sub(r"\s*```$", "", raw_json)

json_resp = json.loads(raw_json)

print(type(json_resp))
print(json_resp)


for item in json_resp:
    print("-" * 50)
    for k, v in item.items():
        print(f"{k}: {v}")



<class 'list'>
[{'name': 'Innovation', 'justification': 'The proposal introduces a novel approach by focusing on the emotional effects of interactions between organic and inorganic social media accounts, which is a relatively unexplored area in digital mental health research.', 'score': 4, 'strengths': 'The innovative focus on the dynamics of social media interactions provides a fresh perspective on mental health interventions.', 'weaknesses': 'While the concept is innovative, the application of existing methodologies may limit the potential for groundbreaking findings.', 'recommendations': 'Consider incorporating more cutting-edge technologies or methodologies, such as machine learning for sentiment analysis, to enhance the innovative aspect.'}, {'name': 'Significance', 'justification': 'The research addresses a critical gap in understanding how social media interactions impact emotional well-being, which is highly relevant given the increasing use of digital platforms for mental heal

In [52]:
# Refactored  (in the code above actually)

display(Markdown(proposal.content))
# display(Markdown(research_plan.content))


Community-Driven Digital Health Platforms for Mental Health Support

1. Introduction  
The rise of social media has transformed the way individuals interact, creating both opportunities and challenges for emotional well-being. As users engage with a mix of organic accounts—real users—and inorganic accounts—bots or automated profiles—the emotional consequences of these interactions have become increasingly significant. Understanding how these dynamics affect sentiments such as anger and fear is crucial for developing effective digital mental health interventions. Given the prevalence of harmful online behaviors, there is an urgent need to explore these interactions to inform strategies that can mitigate negative emotional impacts and promote mental well-being.

2. Problem Statement  
The core problem being addressed is the lack of understanding regarding the emotional effects of interactions between organic social media users and inorganic accounts. This research aims to answer the following questions:  
- How do interactions with inorganic accounts influence the emotional states of organic users?  
- What specific sentiments, such as anger and fear, are most affected by these interactions?  
- How can insights from these interactions inform the development of digital mental health interventions?

3. Objectives  
The main objective of this research is to investigate the causal effects of online interactions on emotional well-being, particularly focusing on the dynamics between organic and inorganic accounts.  
- Specific Objective 1: To quantify the emotional outcomes of interactions with inorganic accounts using a Neyman-Rubin causal model.  
- Specific Objective 2: To identify the specific sentiments that are most impacted by these interactions.  
- Specific Objective 3: To develop recommendations for digital mental health interventions based on the findings.

4. Preliminary Literature Review  
The supporting research paper, "Causal Analysis of Online Outcomes of Interactions between Organic and Inorganic Accounts," provides critical insights into the emotional consequences of online interactions. It highlights the complex relationship between user engagement and emotional outcomes, demonstrating that interactions with inorganic accounts can lead to heightened feelings of anger and fear among organic users. This prior work underscores the necessity of further investigation into these dynamics, motivating the proposed research to explore causal relationships and inform intervention strategies.

5. Methodology  
The research design will employ a mixed-methods approach structured as follows:  

Phase 1: Platform Development  
- Create a digital platform for users to report interactions with inorganic accounts.  
- Integrate tools for tracking emotional responses and sentiment analysis.  

Phase 2: User Engagement and Data Collection  
- Recruit organic social media users to participate in the study.  
- Collect qualitative and quantitative data on user interactions and emotional outcomes.  

Phase 3: Evaluation and Analysis  
- Analyze the data using the Neyman-Rubin causal model to assess emotional impacts.  
- Evaluate the findings to develop actionable recommendations for digital mental health interventions.  

References  
Causal Analysis of Online Outcomes of Interactions between Organic and Inorganic Accounts. Project_Files/Papers/CausalOrgInorgContent.pdf

6. Expected Outcomes  
The expected impact of this research includes a deeper understanding of how online interactions affect emotional well-being, leading to the development of targeted digital mental health interventions. By quantifying the emotional consequences of these interactions, the research aims to provide valuable insights that can help mitigate negative emotional impacts and promote healthier online environments.

7. Budget Justification  
The project will require funding for platform development, user recruitment, data collection tools, and analysis software. Additionally, costs will include personnel for data analysis and participant support, as well as dissemination of findings through publications and community outreach.

8. Conclusion  
In conclusion, this proposal outlines a critical investigation into the effects of online interactions on emotional well-being, with the potential to inform effective digital mental health interventions. By addressing the emotional impacts of interactions between organic and inorganic accounts, this research aims to contribute significantly to the field of digital mental health and promote healthier online experiences for users.